# 01e - Re-export Cities with Inconsistent Box Sizes
Dresden, Leipzig, Munster, and Rostock re-exported with the corrected, consistent bounding box size used by the rest of the dataset. Exports are tracked by task ID, not name.

## Cell 1: Install packages, authenticate Earth Engine

In [8]:
!pip install earthengine-api -q

import ee
ee.Authenticate()
ee.Initialize(project='lulc-sentinel1')

## Cell 2: Mount Drive

In [7]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_FOLDER = '/content/drive/MyDrive/lulc_dataset'
print("Working folder:", DRIVE_FOLDER)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working folder: /content/drive/MyDrive/lulc_dataset


## Cell 3: Define corrected bounding boxes

In [3]:
def box(lon, lat, half_w=0.1, half_h=0.055):
    return ee.Geometry.Rectangle([lon - half_w, lat - half_h, lon + half_w, lat + half_h])

# Centers taken from the original manually-typed rectangles in 01_data_collection.ipynb
city_centers = {
    "Leipzig": (12.39, 51.34),
    "Dresden": (13.75, 51.06),
    "Rostock": (12.125, 54.115),
    "Munster": (7.625, 51.95),
}

fix_cities = {name: box(lon, lat) for name, (lon, lat) in city_centers.items()}

for name, geom in fix_cities.items():
    bounds = geom.bounds().getInfo()['coordinates']
    print(f"{name}: {bounds}")

Leipzig: [[[12.290000000000001, 51.28499999999997], [12.49, 51.28499999999997], [12.49, 51.39504255061836], [12.290000000000001, 51.39504255061836], [12.290000000000001, 51.28499999999997]]]
Dresden: [[[13.65, 51.00499999999997], [13.850000000000001, 51.00499999999997], [13.850000000000001, 51.115042642994794], [13.65, 51.115042642994794], [13.65, 51.00499999999997]]]
Rostock: [[[12.024999999999999, 54.059999999999974], [12.225000000000001, 54.059999999999974], [12.225000000000001, 54.17004141693778], [12.024999999999999, 54.17004141693778], [12.024999999999999, 54.059999999999974]]]
Munster: [[[7.5249999999999995, 51.894999999999975], [7.725000000000001, 51.894999999999975], [7.725000000000001, 52.00504233530862], [7.5249999999999995, 52.00504233530862], [7.5249999999999995, 51.894999999999975]]]


## Cell 4: Export function, tracked by task ID

In [4]:
import calendar

def get_month_range(year, month):
    start = f"{year}-{month:02d}-01"
    last_day = calendar.monthrange(year, month)[1]
    end = f"{year}-{month:02d}-{last_day:02d}"
    return start, end

def export_city_month(city_name, geom, year, month):
    start, end = get_month_range(year, month)

    s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filterBounds(geom) \
        .filterDate(start, end) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW'))

    count = s1.size().getInfo()
    if count == 0:
        print(f"  [SKIP] {city_name} {year}-{month:02d}: no Sentinel-1 images found")
        return None

    s1_image = s1.median().select(['VV', 'VH']).clip(geom)
    worldcover_img = ee.ImageCollection('ESA/WorldCover/v200').first().clip(geom)

    tag = f"{city_name}_{year}_{month:02d}"

    s1_task = ee.batch.Export.image.toDrive(
        image=s1_image,
        description=f"s1_{tag}",
        folder='lulc_dataset',
        fileNamePrefix=f"s1_{tag}",
        region=geom,
        scale=10,
        maxPixels=1e9
    )
    s1_task.start()
    s1_task_id = s1_task.status()['id']

    wc_task = ee.batch.Export.image.toDrive(
        image=worldcover_img,
        description=f"wc_{tag}",
        folder='lulc_dataset',
        fileNamePrefix=f"wc_{tag}",
        region=geom,
        scale=10,
        maxPixels=1e9
    )
    wc_task.start()
    wc_task_id = wc_task.status()['id']

    print(f"  [EXPORT STARTED] {tag} ({count} Sentinel-1 images found) "
          f"s1_id={s1_task_id} wc_id={wc_task_id}")
    return {
        "city": city_name,
        "year": year,
        "month": month,
        "s1_file": f"s1_{tag}.tif",
        "wc_file": f"wc_{tag}.tif",
        "s1_image_count": count,
        "s1_task_id": s1_task_id,
        "wc_task_id": wc_task_id,
    }

## Cell 5: Start exports, four cities, 12 months

In [5]:
MONTHS = [
    (2025, 11), (2025, 12), (2026, 1), (2026, 2), (2026, 3), (2026, 4),
    (2026, 5), (2026, 6), (2026, 7), (2026, 8), (2026, 9), (2026, 10),
]

fix_metadata_records = []
total_cities = len(fix_cities)

for idx, (city_name, geom) in enumerate(fix_cities.items()):
    print(f"\n--- [{idx+1}/{total_cities}] {city_name} ---")
    for year, month in MONTHS:
        record = export_city_month(city_name, geom, year, month)
        if record:
            fix_metadata_records.append(record)

task_ids = []
for rec in fix_metadata_records:
    task_ids.append(rec['s1_task_id'])
    task_ids.append(rec['wc_task_id'])

print(f"\nAll export tasks started. Total city-month pairs queued: {len(fix_metadata_records)}")
print(f"Total task IDs being tracked: {len(task_ids)}")


--- [1/4] Leipzig ---
  [EXPORT STARTED] Leipzig_2025_11 (19 Sentinel-1 images found) s1_id=2OIAC33SRNPZ4EGKQD7OPQUJ wc_id=N4LJUCNHQLZCWFWKRBV46QQA
  [EXPORT STARTED] Leipzig_2025_12 (20 Sentinel-1 images found) s1_id=ETAVIT2IVWNK4HHJ44S3IWO4 wc_id=U43WFYSL5ETYXVY5WVAKNTAL
  [EXPORT STARTED] Leipzig_2026_01 (20 Sentinel-1 images found) s1_id=ZTLMPNMD4XMSRKHIFEKD5FAQ wc_id=XDNSMNP2LJBEILZB7RTLY7YT
  [EXPORT STARTED] Leipzig_2026_02 (18 Sentinel-1 images found) s1_id=XHAON6QOOQU7OAPDNHIUYXL3 wc_id=LZEGXJHKWSNFLYLDZQKY4FUG
  [EXPORT STARTED] Leipzig_2026_03 (20 Sentinel-1 images found) s1_id=VJUCJQZYP7TQOZRQPKKPI3UA wc_id=CD2I2HUL3OYZO7345U6OM6VU
  [EXPORT STARTED] Leipzig_2026_04 (22 Sentinel-1 images found) s1_id=GYAMJUUIGVPJUWEWFLWO7EXA wc_id=IXLEQKSXQPBXUJL4NTQP6JCF
  [EXPORT STARTED] Leipzig_2026_05 (31 Sentinel-1 images found) s1_id=Z7UXQJBAETTHTLGQ6TV2H4L7 wc_id=SYP4JIB3UBZUQX7YY27E36JH
  [EXPORT STARTED] Leipzig_2026_06 (24 Sentinel-1 images found) s1_id=LMA5YZLXEMEIBX5NSFXEWO4F 

## Cell 6: Poll export tasks by task ID

In [6]:
import time

total_tasks = len(task_ids)
check_interval_seconds = 60
check_number = 0

while True:
    check_number += 1
    statuses = ee.data.getTaskStatus(task_ids)

    status_counts = {}
    for s in statuses:
        state = s['state']
        status_counts[state] = status_counts.get(state, 0) + 1

    completed = status_counts.get('COMPLETED', 0)
    failed = status_counts.get('FAILED', 0)
    running = status_counts.get('RUNNING', 0)
    ready = status_counts.get('READY', 0)

    print(f"[Check {check_number}] completed={completed}/{total_tasks} "
          f"running={running} ready={ready} failed={failed}  (status breakdown: {status_counts})")

    if completed + failed >= total_tasks:
        print("\nAll export tasks have finished.")
        if failed > 0:
            print(f"Warning: {failed} task(s) failed. Review before continuing.")
        break

    time.sleep(check_interval_seconds)

[Check 1] completed=0/96 running=1 ready=95 failed=0  (status breakdown: {'RUNNING': 1, 'READY': 95})
[Check 2] completed=1/96 running=1 ready=94 failed=0  (status breakdown: {'COMPLETED': 1, 'RUNNING': 1, 'READY': 94})
[Check 3] completed=2/96 running=1 ready=93 failed=0  (status breakdown: {'COMPLETED': 2, 'RUNNING': 1, 'READY': 93})
[Check 4] completed=2/96 running=1 ready=93 failed=0  (status breakdown: {'COMPLETED': 2, 'RUNNING': 1, 'READY': 93})
[Check 5] completed=2/96 running=1 ready=93 failed=0  (status breakdown: {'COMPLETED': 2, 'RUNNING': 1, 'READY': 93})
[Check 6] completed=2/96 running=1 ready=93 failed=0  (status breakdown: {'COMPLETED': 2, 'RUNNING': 1, 'READY': 93})
[Check 7] completed=3/96 running=1 ready=92 failed=0  (status breakdown: {'COMPLETED': 3, 'RUNNING': 1, 'READY': 92})
[Check 8] completed=4/96 running=1 ready=91 failed=0  (status breakdown: {'COMPLETED': 4, 'RUNNING': 1, 'READY': 91})
[Check 9] completed=5/96 running=1 ready=90 failed=0  (status breakdown:

[Check 45] completed=35/96 running=1 ready=60 failed=0  (status breakdown: {'COMPLETED': 35, 'RUNNING': 1, 'READY': 60})
[Check 46] completed=36/96 running=1 ready=59 failed=0  (status breakdown: {'COMPLETED': 36, 'RUNNING': 1, 'READY': 59})
[Check 47] completed=36/96 running=1 ready=59 failed=0  (status breakdown: {'COMPLETED': 36, 'RUNNING': 1, 'READY': 59})
[Check 48] completed=37/96 running=1 ready=58 failed=0  (status breakdown: {'COMPLETED': 37, 'RUNNING': 1, 'READY': 58})
[Check 49] completed=38/96 running=1 ready=57 failed=0  (status breakdown: {'COMPLETED': 38, 'RUNNING': 1, 'READY': 57})
[Check 50] completed=39/96 running=1 ready=56 failed=0  (status breakdown: {'COMPLETED': 39, 'RUNNING': 1, 'READY': 56})
[Check 51] completed=40/96 running=1 ready=55 failed=0  (status breakdown: {'COMPLETED': 40, 'RUNNING': 1, 'READY': 55})
[Check 52] completed=41/96 running=1 ready=54 failed=0  (status breakdown: {'COMPLETED': 41, 'RUNNING': 1, 'READY': 54})
[Check 53] completed=42/96 runni

[Check 80] completed=65/96 running=1 ready=30 failed=0  (status breakdown: {'COMPLETED': 65, 'RUNNING': 1, 'READY': 30})
[Check 81] completed=66/96 running=1 ready=29 failed=0  (status breakdown: {'COMPLETED': 66, 'RUNNING': 1, 'READY': 29})
[Check 82] completed=67/96 running=1 ready=28 failed=0  (status breakdown: {'COMPLETED': 67, 'RUNNING': 1, 'READY': 28})
[Check 83] completed=68/96 running=1 ready=27 failed=0  (status breakdown: {'COMPLETED': 68, 'RUNNING': 1, 'READY': 27})
[Check 84] completed=68/96 running=1 ready=27 failed=0  (status breakdown: {'COMPLETED': 68, 'RUNNING': 1, 'READY': 27})
[Check 85] completed=69/96 running=1 ready=26 failed=0  (status breakdown: {'COMPLETED': 69, 'RUNNING': 1, 'READY': 26})
[Check 86] completed=70/96 running=1 ready=25 failed=0  (status breakdown: {'COMPLETED': 70, 'RUNNING': 1, 'READY': 25})
[Check 87] completed=70/96 running=1 ready=25 failed=0  (status breakdown: {'COMPLETED': 70, 'RUNNING': 1, 'READY': 25})
[Check 88] completed=71/96 runni

[Check 91] completed=74/96 running=1 ready=21 failed=0  (status breakdown: {'COMPLETED': 74, 'RUNNING': 1, 'READY': 21})
[Check 92] completed=75/96 running=1 ready=20 failed=0  (status breakdown: {'COMPLETED': 75, 'RUNNING': 1, 'READY': 20})
[Check 93] completed=76/96 running=1 ready=19 failed=0  (status breakdown: {'COMPLETED': 76, 'RUNNING': 1, 'READY': 19})
[Check 94] completed=77/96 running=1 ready=18 failed=0  (status breakdown: {'COMPLETED': 77, 'RUNNING': 1, 'READY': 18})
[Check 95] completed=78/96 running=1 ready=17 failed=0  (status breakdown: {'COMPLETED': 78, 'RUNNING': 1, 'READY': 17})
[Check 96] completed=79/96 running=1 ready=16 failed=0  (status breakdown: {'COMPLETED': 79, 'RUNNING': 1, 'READY': 16})
[Check 97] completed=80/96 running=1 ready=15 failed=0  (status breakdown: {'COMPLETED': 80, 'RUNNING': 1, 'READY': 15})
[Check 98] completed=80/96 running=1 ready=15 failed=0  (status breakdown: {'COMPLETED': 80, 'RUNNING': 1, 'READY': 15})
[Check 99] completed=81/96 runni

## Cell 7: Verify file dimensions after Drive sync

In [1]:
!pip install earthengine-api rasterio -q

import ee, time, os
ee.Authenticate()
ee.Initialize(project='lulc-sentinel1')

from google.colab import drive
drive.mount('/content/drive')
DRIVE_FOLDER = '/content/drive/MyDrive/lulc_dataset'

city_centers = {
    "Leipzig": (12.39, 51.34),
    "Dresden": (13.75, 51.06),
    "Rostock": (12.125, 54.115),
    "Munster": (7.625, 51.95),
}

# Reconstructed exactly from your Cell 5 output, nothing re-submitted, nothing new created.
_raw_ids = """
Leipzig 2025 11 2OIAC33SRNPZ4EGKQD7OPQUJ N4LJUCNHQLZCWFWKRBV46QQA
Leipzig 2025 12 ETAVIT2IVWNK4HHJ44S3IWO4 U43WFYSL5ETYXVY5WVAKNTAL
Leipzig 2026 1 ZTLMPNMD4XMSRKHIFEKD5FAQ XDNSMNP2LJBEILZB7RTLY7YT
Leipzig 2026 2 XHAON6QOOQU7OAPDNHIUYXL3 LZEGXJHKWSNFLYLDZQKY4FUG
Leipzig 2026 3 VJUCJQZYP7TQOZRQPKKPI3UA CD2I2HUL3OYZO7345U6OM6VU
Leipzig 2026 4 GYAMJUUIGVPJUWEWFLWO7EXA IXLEQKSXQPBXUJL4NTQP6JCF
Leipzig 2026 5 Z7UXQJBAETTHTLGQ6TV2H4L7 SYP4JIB3UBZUQX7YY27E36JH
Leipzig 2026 6 LMA5YZLXEMEIBX5NSFXEWO4F APSXZKTKRAM3D6NSDE3VAYRO
Leipzig 2026 7 BUMMUITN6IVLK4FKC36ANF53 NTJSE5CRGT4Q5NYVKEFAWUPN
Leipzig 2026 8 B5RQOYDU6AH7DRPAK5GVLIC4 U5SOC76DHHAX65EMJGVGHWO6
Leipzig 2026 9 27LSKZDC3I4XWFSI6TI2RQOG A2HBDHKMNZ474FGOKKGBJMAX
Leipzig 2026 10 67YLLV5V7KHYXZNQXVDCVGNC GOJQNNKB3IJOXDMLXYIG3LTC
Dresden 2025 11 EACVYQ24EBSVYBNBJWMETMGD XA2YVAYAMHONK64WPQI33J2F
Dresden 2025 12 C3KNKBUQ3ED36Y2JQGDZZYMH QDNGQIS3NCK2MJG3Y3FU77AS
Dresden 2026 1 XZTJBIC7THFV45NDU7GV2INT 67HY3CPWITBWZDZYPJCHYZK4
Dresden 2026 2 E34F3CL3QWKMTQSZCGSPWVSC 6B2OU42OWNEVD6ASP5FQPDHT
Dresden 2026 3 C6BS4NYSWZ3WJYU7YXHB4AS6 VBN2SSNQGYK5IYNE44N5NAFU
Dresden 2026 4 W7HU6L4SPEWET22U7R4VZE57 XX4QOELNLVJP5R5EHK3J2PNE
Dresden 2026 5 C2GMZGFWKMCNRP5BYSBQMPQR SU3AKRSUEUZCJ5MP5JJHC5SE
Dresden 2026 6 IYRZSWNPSZBZEORT2M7DN2QS GPYQQFO2PZA5VJR4T5B7MVUX
Dresden 2026 7 2EXWS2PKDH7S4R4RAH7VYWCH 6VCWTJZZLQWVHDJOPWKMRIGD
Dresden 2026 8 JNEH7PTK3X5Z5JDVC6JAUXB2 COKLZ23NTMXSHXIA3F7O5JPA
Dresden 2026 9 BSLZKQEUQS6SPPONRA76BI6U NLXQDU7QQ6EIF3O3KB6JQJN7
Dresden 2026 10 NAW57DOHTH5CLXWNDT3XDYBT OU36K25FXZSS2PDUZ3I4HWME
Rostock 2025 11 X7PEGNQTKR6SSHXVWFLBRZHQ OI5RGOAV5VD2EWP6P3UEPDF5
Rostock 2025 12 M2UNIUKO3LWKKBZKXTPRHFMN IEBVKSANFPTWMWGEEE64VLUT
Rostock 2026 1 QAYJQE7K4C5VB47LZ4NEIMI2 FTHOMIV2QTTX6FV5EVXQ47PJ
Rostock 2026 2 4J3WGYNWMH4U2F4GTVLDBGDL CHHCVZ7VY42YSL7VYY2RTTIT
Rostock 2026 3 6LW2BFIRNFBHFQSDVXD4CA6T S6SJFJQIYAQSFP22XEXBB7LU
Rostock 2026 4 CWVIWPF7LBKP5J7ER66XGBFD 7QIVFXUP6KZPZEVRIKT5GSQJ
Rostock 2026 5 CJKLDTFMRSWIYW7A5UPYIF5B GH4O5MC3I4D2UZDK7F4IV7IN
Rostock 2026 6 PE4FDYZACZKIF7LHV5DOFGBC X7WEIIAIEFLOD3JKADQBGTXK
Rostock 2026 7 QCLRBH77LJRTNJX73JFPXLYG OAEHMEB364HC5UOOBERYC2GI
Rostock 2026 8 OP2VWVTANPT4XUXEL7H7ULWC RMEUE7XSCYFYKY3MSCXXXNEV
Rostock 2026 9 LQU7CSA2UKLF6H3E43KSK3C5 JJVEPP7HN2RQW42N2AI5PQDH
Rostock 2026 10 MDUCDROCSTB7H4DT7OEILRBT PBRPQFLVP5TJQPSF2TNH227P
Munster 2025 11 EDXEMUZSMP3BEIRPRUBYCYGD M6RPY4IHZPAPUBI4ENYXGCAI
Munster 2025 12 TMGNR5L5GCLOYZC7Q24PYIFR DH5LFKXNXYQKY47O4ZTXZDB7
Munster 2026 1 Q4VCQOEUWOGY24PGCETVIYYD NFKH74HB5WZV5JK5MEDMT364
Munster 2026 2 IGS63X3SF5FKMUOM72VRMG7G A5TP2IN5ONADSCMKXXTWMMQR
Munster 2026 3 J64WSZHNUAAYK2V4K7NYSCIR FKLF67XNXLXWLK3EL2A47HTI
Munster 2026 4 UM4IZKSOIJAMIOAZBVNPOXR2 NRRYPRJ7MUIIIK3QC3BYZEQ3
Munster 2026 5 JAKV72LTJBK35UAEP63PLPH2 IMTNL66K3VLRZQHBJOOEHHPQ
Munster 2026 6 AQE3BBITB42NTN2URX72BOKN 5YN43C7J5M6RET7PJZI6J5LO
Munster 2026 7 UOAJ32FA6QHRZ7ZOSSDG2KET BFJ2BCHM3GUT7JWSOFDC4CYU
Munster 2026 8 KKPN3KCDMW2HW7FWD6SPTAP6 KKZJT5QUO6KKIDDDYHV4JKL3
Munster 2026 9 Y7C246WDOQS6UTOPPTBCLPMC ZI7PNQK2ATN35XRHZB2CATAC
Munster 2026 10 SRFQV5YBIGG6EIO6PNU72L46 SDYZBPKSYHBXO6W6QRKRLBM4
""".strip().splitlines()

fix_metadata_records = []
for line in _raw_ids:
    city, year, month, s1_id, wc_id = line.split()
    year, month = int(year), int(month)
    tag = f"{city}_{year}_{month:02d}"
    fix_metadata_records.append({
        "city": city, "year": year, "month": month,
        "s1_file": f"s1_{tag}.tif", "wc_file": f"wc_{tag}.tif",
        "s1_task_id": s1_id, "wc_task_id": wc_id,
    })

task_ids = []
for rec in fix_metadata_records:
    task_ids.append(rec['s1_task_id'])
    task_ids.append(rec['wc_task_id'])

fix_cities = list(city_centers.keys())

print(f"Reconstructed {len(fix_metadata_records)} city-month records and {len(task_ids)} task IDs.")
print("Nothing was re-submitted. Checking their REAL current status now:\n")

statuses = ee.data.getTaskStatus(task_ids)
status_counts = {}
for s in statuses:
    state = s['state']
    status_counts[state] = status_counts.get(state, 0) + 1

print(f"Current status breakdown: {status_counts}")
completed = status_counts.get('COMPLETED', 0)
failed = status_counts.get('FAILED', 0)
total_tasks = len(task_ids)
print(f"completed={completed}/{total_tasks} failed={failed}")

if completed + failed >= total_tasks:
    print("\nAll tasks already finished while the runtime was down. Proceed straight to the verification step (old Cell 7's dimension check).")
else:
    print(f"\n{total_tasks - completed - failed} task(s) still in progress. Run the polling loop (old Cell 6 logic) below to wait for them.")
    check_interval_seconds = 60
    check_number = 0
    while True:
        check_number += 1
        statuses = ee.data.getTaskStatus(task_ids)
        status_counts = {}
        for s in statuses:
            state = s['state']
            status_counts[state] = status_counts.get(state, 0) + 1
        completed = status_counts.get('COMPLETED', 0)
        failed = status_counts.get('FAILED', 0)
        running = status_counts.get('RUNNING', 0)
        ready = status_counts.get('READY', 0)
        print(f"[Check {check_number}] completed={completed}/{total_tasks} running={running} ready={ready} failed={failed}  (status breakdown: {status_counts})")
        if completed + failed >= total_tasks:
            print("\nAll export tasks have finished.")
            break
        time.sleep(check_interval_seconds)

Mounted at /content/drive
Reconstructed 48 city-month records and 96 task IDs.
Nothing was re-submitted. Checking their REAL current status now:

Current status breakdown: {'COMPLETED': 96}
completed=96/96 failed=0

All tasks already finished while the runtime was down. Proceed straight to the verification step (old Cell 7's dimension check).


In [3]:
import rasterio
import time

def with_retries(func, *args, retries=5, delay=15, **kwargs):
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            last_err = e
            print(f"  Attempt {attempt}/{retries} failed: {e}")
            if attempt < retries:
                if attempt >= retries - 1:
                    print("  Remounting Drive before next attempt...")
                    drive.mount('/content/drive', force_remount=True)
                time.sleep(delay)
    raise last_err

def get_shape(path):
    with rasterio.open(path) as src:
        return (src.height, src.width)

all_clean = True
city_shapes = {}

for city in fix_cities:
    recs = [r for r in fix_metadata_records if r['city'] == city]
    s1_shapes_seen = set()
    wc_shapes_seen = set()
    for rec in recs:
        s1_path = f"{DRIVE_FOLDER}/{rec['s1_file']}"
        wc_path = f"{DRIVE_FOLDER}/{rec['wc_file']}"
        s1_shapes_seen.add(with_retries(get_shape, s1_path))
        wc_shapes_seen.add(with_retries(get_shape, wc_path))

    city_shapes[city] = (s1_shapes_seen, wc_shapes_seen)
    s1_ok = len(s1_shapes_seen) == 1
    wc_ok = len(wc_shapes_seen) == 1
    print(f"{city}: s1 shapes={s1_shapes_seen} ({'OK' if s1_ok else 'INCONSISTENT'})  "
          f"wc shapes={wc_shapes_seen} ({'OK' if wc_ok else 'INCONSISTENT'})")
    if not (s1_ok and wc_ok):
        all_clean = False

print()
if all_clean:
    print("All four cities: s1 is consistent across all 12 months, wc is consistent across all 12 months. Safe to proceed to Cell 8.")
else:
    print("Real inconsistency found. Do not proceed.")

  Attempt 1/5 failed: /content/drive/MyDrive/lulc_dataset/s1_Leipzig_2026_04.tif: Transport endpoint is not connected
Leipzig: s1 shapes={(1225, 2228)} (OK)  wc shapes={(1225, 2227)} (OK)
Dresden: s1 shapes={(1225, 2227)} (OK)  wc shapes={(1225, 2228)} (OK)
Rostock: s1 shapes={(1225, 2228)} (OK)  wc shapes={(1225, 2227)} (OK)
Munster: s1 shapes={(1226, 2228)} (OK)  wc shapes={(1226, 2228)} (OK)

All four cities: s1 is consistent across all 12 months, wc is consistent across all 12 months. Safe to proceed to Cell 8.


## Cell 8: Clear stale records, add corrected metadata

In [4]:
import pandas as pd
import os

metadata_path = f"{DRIVE_FOLDER}/dataset_metadata.csv"
patch_metadata_path = f"{DRIVE_FOLDER}/patch_metadata.csv"
fixed_cities = ['Leipzig', 'Dresden', 'Rostock', 'Munster']

metadata_df = pd.read_csv(metadata_path)
print("dataset_metadata.csv rows for the fixed cities (filenames unchanged, just confirming):")
print(metadata_df[metadata_df['city'].isin(fixed_cities)][['city', 'year', 'month', 's1_file', 'wc_file']])

if os.path.exists(patch_metadata_path):
    patch_df = pd.read_csv(patch_metadata_path)
    before = len(patch_df)
    patch_df = patch_df[~patch_df['city'].isin(fixed_cities)]
    after = len(patch_df)
    patch_df.to_csv(patch_metadata_path, index=False)
    print(f"\npatch_metadata.csv: removed {before - after} stale patch rows for {fixed_cities}.")
    print(f"{after} rows remain for other cities, untouched.")
else:
    print("\nNo patch_metadata.csv found yet, nothing to clear.")

print("\nNext step: re-run 02_preprocessing_28cities_v3.ipynb from Cell 1.")

dataset_metadata.csv rows for the fixed cities (filenames unchanged, just confirming):
        city  year  month                 s1_file                 wc_file
336  Leipzig  2025     11  s1_Leipzig_2025_11.tif  wc_Leipzig_2025_11.tif
337  Leipzig  2025     12  s1_Leipzig_2025_12.tif  wc_Leipzig_2025_12.tif
338  Leipzig  2026      1  s1_Leipzig_2026_01.tif  wc_Leipzig_2026_01.tif
339  Leipzig  2026      2  s1_Leipzig_2026_02.tif  wc_Leipzig_2026_02.tif
340  Leipzig  2026      3  s1_Leipzig_2026_03.tif  wc_Leipzig_2026_03.tif
341  Leipzig  2026      4  s1_Leipzig_2026_04.tif  wc_Leipzig_2026_04.tif
342  Leipzig  2026      5  s1_Leipzig_2026_05.tif  wc_Leipzig_2026_05.tif
343  Leipzig  2026      6  s1_Leipzig_2026_06.tif  wc_Leipzig_2026_06.tif
344  Leipzig  2026      7  s1_Leipzig_2026_07.tif  wc_Leipzig_2026_07.tif
345  Leipzig  2026      8  s1_Leipzig_2026_08.tif  wc_Leipzig_2026_08.tif
346  Leipzig  2026      9  s1_Leipzig_2026_09.tif  wc_Leipzig_2026_09.tif
347  Leipzig  2026     10